# Kruskal's Algorithm: Minimum Spanning Tree

Kruskal's algorithm finds the cheapest way to connect every node in a weighted, undirected graph.

Minimum spanning trees model a practical engineering question: how do you connect many sites without paying for redundant links? The same structure appears in cable layout, road planning, circuit design, clustering, and network backbone design.

The result is a **minimum spanning tree**:

- **spanning**: reaches every node
- **tree**: has no cycles
- **minimum**: has the lowest possible total edge cost

<details>
<summary>Big idea</summary>

Sort edges from cheapest to most expensive. Keep an edge if it connects two separate groups. Skip it if it would create a cycle.

</details>

## 1. The Mental Model

Imagine a festival map where each site needs power. Cables have different prices. We want to connect every site without paying for extra loops.

Kruskal's algorithm acts like a careful builder:

1. Sort all possible cables by cost.
2. Look at the cheapest remaining cable.
3. Add it if it connects two separate groups.
4. Skip it if those sites are already connected.
5. Stop when every site is connected.

<details>
<summary>Why cycles are wasteful</summary>

If a cable creates a cycle, there is already another path between those two sites. Keeping it would add cost without helping reach a new site.

</details>

## 2. Build the Objects

Implementation plan:

1. `Site` names one node in the graph.
2. `Cable` connects two sites with a cost.
3. `UnionFind` tracks which sites are already in the same connected group.
4. `KruskalStep` records each edge decision.
5. `KruskalRunner` sorts edges and builds the tree.
6. `MSTReplay` prints the decisions.

<details>
<summary>Union-find hint</summary>

Union-find is a fast way to answer: "Are these two nodes already connected?" If yes, adding their edge would create a cycle.

</details>

**Object model.** Define `Site`, `Cable`, the named objects used by the next examples.


In [ ]:
from dataclasses import dataclass

from collections import defaultdict

@dataclass(frozen=True)
class Site:
    name: str

    def __str__(self) -> str:
        return self.name

@dataclass(frozen=True)
class Cable:
    left: Site
    right: Site
    cost: int

    def __str__(self) -> str:
        return f"{self.left}-{self.right} (${self.cost})"


**Object model.** Define `UnionFind`, the named objects used by the next examples.


In [ ]:
class UnionFind:
    def __init__(self, sites: list[Site]):
        self.parent = {site: site for site in sites}
        self.rank = {site: 0 for site in sites}

    def find(self, site: Site) -> Site:
        if self.parent[site] != site:
            self.parent[site] = self.find(self.parent[site])
        return self.parent[site]

    def connected(self, first: Site, second: Site) -> bool:
        return self.find(first) == self.find(second)

    def union(self, first: Site, second: Site) -> bool:
        first_root = self.find(first)
        second_root = self.find(second)

        if first_root == second_root:
            return False

        if self.rank[first_root] < self.rank[second_root]:
            first_root, second_root = second_root, first_root

        self.parent[second_root] = first_root

        if self.rank[first_root] == self.rank[second_root]:
            self.rank[first_root] += 1

        return True

    def groups(self) -> tuple[tuple[str, ...], ...]:
        grouped_sites: dict[Site, list[str]] = defaultdict(list)
        for site in self.parent:
            grouped_sites[self.find(site)].append(site.name)

        return tuple(
            sorted(tuple(sorted(names)) for names in grouped_sites.values())
        )


**Trace model.** Define `KruskalStep`, `MSTResult`, the structure used to capture replayable algorithm state.


In [ ]:
@dataclass
class KruskalStep:
    round_number: int
    cable: Cable
    action: str
    reason: str
    components: tuple[tuple[str, ...], ...]
    chosen_cables: tuple[Cable, ...]
    total_cost: int

@dataclass
class MSTResult:
    chosen_cables: list[Cable]
    total_cost: int
    steps: list[KruskalStep]
    connected: bool


**Algorithm engine.** Define `KruskalRunner`, the class that runs the main simulation or algorithm.


In [ ]:
class KruskalRunner:
    def __init__(self, sites: list[Site], cables: list[Cable]):
        self.sites = sites
        self.cables = cables

    def run(self) -> MSTResult:
        union_find = UnionFind(self.sites)
        sorted_cables = sorted(
            self.cables,
            key=lambda cable: (cable.cost, cable.left.name, cable.right.name),
        )
        chosen_cables: list[Cable] = []
        steps: list[KruskalStep] = []
        total_cost = 0

        for round_number, cable in enumerate(sorted_cables, start=1):
            if union_find.connected(cable.left, cable.right):
                action = "skip"
                reason = "would create a cycle"
            else:
                union_find.union(cable.left, cable.right)
                chosen_cables.append(cable)
                total_cost += cable.cost
                action = "take"
                reason = "connects two separate groups"

            steps.append(
                KruskalStep(
                    round_number=round_number,
                    cable=cable,
                    action=action,
                    reason=reason,
                    components=union_find.groups(),
                    chosen_cables=tuple(chosen_cables),
                    total_cost=total_cost,
                )
            )

            if len(chosen_cables) == len(self.sites) - 1:
                break

        return MSTResult(
            chosen_cables=chosen_cables,
            total_cost=total_cost,
            steps=steps,
            connected=len(chosen_cables) == len(self.sites) - 1,
        )


## 3. Create the Festival Cable Map

Each site needs to be connected. Each cable has a cost. The graph is undirected, so `Arcade-Bakery` is the same connection as `Bakery-Arcade`.

<details>
<summary>What the algorithm sees</summary>

Kruskal does not start from one node. It looks globally at all edges, cheapest first.

</details>

In [2]:
arcade = Site("Arcade")
bakery = Site("Bakery")
cinema = Site("Cinema")
dock = Site("Dock")
exhibit = Site("Exhibit")
garden = Site("Garden")

festival_sites = [arcade, bakery, cinema, dock, exhibit, garden]
festival_cables = [
    Cable(bakery, cinema, 2),
    Cable(exhibit, garden, 2),
    Cable(arcade, bakery, 3),
    Cable(cinema, dock, 3),
    Cable(arcade, cinema, 4),
    Cable(dock, exhibit, 4),
    Cable(bakery, dock, 5),
    Cable(cinema, exhibit, 7),
    Cable(arcade, garden, 10),
]

print("Candidate cables, cheapest first:")
for cable in sorted(festival_cables, key=lambda cable: (cable.cost, cable.left.name, cable.right.name)):
    print(f"  {cable}")

Candidate cables, cheapest first:
  Bakery-Cinema ($2)
  Exhibit-Garden ($2)
  Arcade-Bakery ($3)
  Cinema-Dock ($3)
  Arcade-Cinema ($4)
  Dock-Exhibit ($4)
  Bakery-Dock ($5)
  Cinema-Exhibit ($7)
  Arcade-Garden ($10)


## 4. Run Kruskal's Algorithm

Kruskal sorts all cables and makes one decision at a time: take it or skip it.

<details>
<summary>Stopping rule</summary>

A tree with `n` nodes has exactly `n - 1` edges. Once Kruskal has chosen that many cables, the minimum spanning tree is complete.

</details>

In [3]:
runner = KruskalRunner(festival_sites, festival_cables)
result = runner.run()

print("Minimum spanning tree:")
for cable in result.chosen_cables:
    print(f"  take {cable}")

print(f"\nTotal cost: ${result.total_cost}")
print(f"Connected all sites: {result.connected}")

Minimum spanning tree:
  take Bakery-Cinema ($2)
  take Exhibit-Garden ($2)
  take Arcade-Bakery ($3)
  take Cinema-Dock ($3)
  take Dock-Exhibit ($4)

Total cost: $14
Connected all sites: True


## 5. Replay the Decisions

The key moment in Kruskal is the skip: a cheap-looking edge can still be rejected if it creates a cycle.

<details>
<summary>Reading the components</summary>

Each component is a group of sites already connected by chosen cables. Kruskal keeps merging components until only one group remains.

</details>

In [4]:
class MSTReplay:
    def __init__(self, result: MSTResult):
        self.result = result

    def show(self) -> None:
        for step in self.result.steps:
            action = "TAKE" if step.action == "take" else "SKIP"
            components = " | ".join("{" + ", ".join(group) + "}" for group in step.components)
            chosen = ", ".join(str(cable) for cable in step.chosen_cables) or "none yet"

            print(f"Round {step.round_number}: {action} {step.cable}")
            print(f"  reason: {step.reason}")
            print(f"  components: {components}")
            print(f"  chosen: {chosen}")
            print(f"  total cost: ${step.total_cost}\n")


MSTReplay(result).show()

Round 1: TAKE Bakery-Cinema ($2)
  reason: connects two separate groups
  components: {Arcade} | {Bakery, Cinema} | {Dock} | {Exhibit} | {Garden}
  chosen: Bakery-Cinema ($2)
  total cost: $2

Round 2: TAKE Exhibit-Garden ($2)
  reason: connects two separate groups
  components: {Arcade} | {Bakery, Cinema} | {Dock} | {Exhibit, Garden}
  chosen: Bakery-Cinema ($2), Exhibit-Garden ($2)
  total cost: $4

Round 3: TAKE Arcade-Bakery ($3)
  reason: connects two separate groups
  components: {Arcade, Bakery, Cinema} | {Dock} | {Exhibit, Garden}
  chosen: Bakery-Cinema ($2), Exhibit-Garden ($2), Arcade-Bakery ($3)
  total cost: $7

Round 4: TAKE Cinema-Dock ($3)
  reason: connects two separate groups
  components: {Arcade, Bakery, Cinema, Dock} | {Exhibit, Garden}
  chosen: Bakery-Cinema ($2), Exhibit-Garden ($2), Arcade-Bakery ($3), Cinema-Dock ($3)
  total cost: $10

Round 5: SKIP Arcade-Cinema ($4)
  reason: would create a cycle
  components: {Arcade, Bakery, Cinema, Dock} | {Exhibit, Gard

## 6. Experiments

Try changing the graph and watch the minimum spanning tree change.

<details>
<summary>Experiment hint</summary>

Kruskal is greedy, but not careless. It loves cheap edges only when they connect new components.

</details>

In [5]:
shortcut_cables = festival_cables + [Cable(arcade, garden, 1)]
shortcut_result = KruskalRunner(festival_sites, shortcut_cables).run()

print("After adding a cheap Arcade-Garden shortcut:")
for cable in shortcut_result.chosen_cables:
    print(f"  take {cable}")

print(f"\nOriginal cost: ${result.total_cost}")
print(f"Shortcut cost: ${shortcut_result.total_cost}")

After adding a cheap Arcade-Garden shortcut:
  take Arcade-Garden ($1)
  take Bakery-Cinema ($2)
  take Exhibit-Garden ($2)
  take Arcade-Bakery ($3)
  take Cinema-Dock ($3)

Original cost: $14
Shortcut cost: $11


In [6]:
workshop = Site("Workshop")
disconnected_sites = festival_sites + [workshop]
disconnected_result = KruskalRunner(disconnected_sites, festival_cables).run()

print("Disconnected graph experiment:")
print(f"Connected all sites: {disconnected_result.connected}")
print(f"Edges chosen: {len(disconnected_result.chosen_cables)}")
print(f"Needed for a spanning tree: {len(disconnected_sites) - 1}")

print("\nChosen forest:")
for cable in disconnected_result.chosen_cables:
    print(f"  {cable}")

Disconnected graph experiment:
Connected all sites: False
Edges chosen: 5
Needed for a spanning tree: 6

Chosen forest:
  Bakery-Cinema ($2)
  Exhibit-Garden ($2)
  Arcade-Bakery ($3)
  Cinema-Dock ($3)
  Dock-Exhibit ($4)


## What You Should Remember

Kruskal's algorithm is a greedy MST algorithm:

- Sort edges by cost.
- Take the cheapest edge that connects two different components.
- Skip edges that create cycles.
- Use union-find to track connected components quickly.
- If the graph is disconnected, Kruskal returns a minimum spanning forest, not a full spanning tree.

<details>
<summary>Where this shows up</summary>

Minimum spanning trees show up in network design, clustering, circuit layout, road planning, and any problem where you want to connect everything with minimum total cost.

</details>

## Visual Trace + Rigor Studio

**Problem frame.** Connect all vertices with minimum total edge cost.

**Interactive animation target.** Animate sorted candidate edges, accepted edges, rejected cycles, and connected components.

**Correctness handle.** By the cut property, the cheapest safe edge across a cut can be part of an MST.

**Complexity handle.** O(E log E) for Kruskal plus near-constant union-find operations.

**Failure mode to test.** A disconnected graph has a minimum spanning forest, not a single spanning tree.

**Studio task.** Change one edge weight and identify the first decision in the trace that changes.


In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "courseware").exists():
        sys.path.insert(0, str(candidate))
        break

from courseware import AlgorithmPlayer, AlgorithmTrace, TraceStep, render_trace_table

# Convert the implementation above into snapshots:
# trace = AlgorithmTrace("Topic trace")
# trace.append("start", {"your_state": ...}, "What changed?", invariant="What remains true?")
# AlgorithmPlayer(trace, your_renderer).display()
print("Use AlgorithmTrace to turn this notebook's algorithm into a step-by-step visual player.")
